In [1]:
from probjax.nn.tokenizer import Tokenizer, GaussianFourierEmbedding
from probjax.nn import Transformer
import jax.numpy as jnp
import jax
import haiku as hk

from jax import Array
from jaxtyping import PyTree
from typing import List, Tuple, Dict, Any, Optional, Callable, Union

/root/miniconda3/envs/probjax/lib/python3.10/site-packages/jax/_src/api_util.py:174: SyntaxWarning: Jitted function has static_argnums=(3, 4), but only accepts 4 positional arguments. This warning will be replaced by an error after 2022-08-20 at the earliest.
  warnings.warn(f"Jitted function has {argnums_name}={argnums}, "


In [19]:
dim_per_id = jnp.array([1, 2])
node_id = jnp.arange(2)
x = jnp.array([1, 2, 3])

embed = lambda x: x.sum()

@jax.jit
def f(x: Array, node_id) -> Array:
    dims = jax.lax.map(lambda i: dim_per_id[i], node_id)
    return dims

In [20]:
f(x, node_id)

Array([1, 2], dtype=int32)

In [3]:
def pad_last_dim(value, dim, mode="edge"):
    return jnp.pad(value, [(0, 0) for i in range(len(value.shape)-1)] + [(0,dim-value.shape[-1])], mode=mode)

In [5]:

class StructuredTokenizer(Tokenizer):
    def __init__(
        self,
        output_dim: int,
        max_sequence_length: int,
        data_name_to_id: dict[str, int],
        value_embeding_builder: Optional[Callable] = None,
        node_embeding_builder: Optional[Callable] = None,
        node_meta_data_embeding_builder: Optional[Callable] = None,
        distributor: Optional[Union[Callable, str]] = "equal",
        accummulator: Optional[Union[Callable, str]] = "concat",
        learn_node_embeding: bool = True,
        learn_value_embeding: bool = False,
        learn_meta_data_embeding: bool = False,
        name: str | None = "tokenizer",
    ):
        self.max_sequence_length = max_sequence_length
        self.data_name_to_id = data_name_to_id
        super().__init__(
            output_dim,
            node_embeding_builder,
            value_embeding_builder,
            node_meta_data_embeding_builder,
            distributor,
            accummulator,
            learn_node_embeding,
            learn_value_embeding,
            learn_meta_data_embeding,
            name,
        )

    def __call__(self, data: dict[str,Array], meta_data: Optional[PyTree] = None):
        output_dim1, output_dim2, output_dim3 = self.distribute_output_dim(
            with_meta_data=meta_data is not None
        )
        
        data_id = jnp.array([self.data_name_to_id[k] for k in data.keys()])
        data_id_embeding = self.node_embeding(data_id, output_dim1)
        value_embeding = self.value_embeding(data, output_dim2)
        if meta_data is not None:
            meta_data_embeding = self.meta_data_embeding(meta_data, output_dim3)
        else:
            meta_data_embeding = None
        
        if meta_data_embeding is  None:
            data_id_embeding, value_embeding = jnp.broadcast_arrays(
                data_id_embeding, value_embeding
            )
        else:
            data_id_embeding, value_embeding, meta_data_embeding = jnp.broadcast_arrays(
                data_id_embeding, value_embeding, meta_data_embeding
            )
        
        tokens = self.accumulate(data_id_embeding, value_embeding, meta_data_embeding)
        
        return tokens
        
        
    @hk.transparent
    def value_embeding(self, value, output_dim):
        if self.value_embeding_builder is None:
            value_embeding_fns = {}
            for k in value:
                value_embeding_fns[k] = hk.Linear(
                    output_dim,
                )
        else:
            value_embeding_fns = self.value_embeding_builder(id, value, output_dim)

        tokens = []
        for k in value:
            out = value_embeding_fns[k](value[k])
            if self.learn_value_embeding:
                out = jax.lax.stop_gradient(out)
            tokens.append(out[:, None, :])
            
        tokens = jnp.concatenate(tokens, axis=-2)
        return tokens

    @hk.transparent
    def node_embeding(self, node, output_dim):
        if self.node_embeding_builder is None:
            node_embeding_fn = hk.Embed(
                self.max_sequence_length,
                output_dim,
                w_init=hk.initializers.Orthogonal(scale=0.5),
            )
        else:
            node_embeding_fn = self.node_embeding_builder(output_dim)

        out = node_embeding_fn(node)
        if self.learn_node_embeding:
            out = jax.lax.stop_gradient(out)
        return out


In [6]:


def f(data):
    embedding_nets = {}
    outs = []
    for key, value in data.items():
        embedding_nets[key] = hk.Linear(10)   
        out = embedding_nets[key](value)
        outs.append(out)
    return jnp.concatenate(outs, axis=-1)
        
    
init_fn, apply_fn = hk.without_apply_rng(hk.transform(f))
    

In [7]:
from probjax.nn.tokenizer import StructuredTokenizer

In [8]:
node_id = jnp.array([0,1,2])
values = (jnp.ones((10,1,)), jnp.ones((10,10,)), jnp.ones((10,100,)))

In [9]:
data = {"x1": values[0], "x2": values[1], "x3": values[2]}
data_sub = {"x1": values[0], "x2": values[1]}
data_name_to_id = {"x1": 0, "x2": 1, "x3": 2}
params = init_fn(jax.random.PRNGKey(42), data)  


In [10]:
def f(value):
    tokenizer = StructuredTokenizer(10,3, data_name_to_id)
    tokens = tokenizer(value)
    return tokens

init_fn, apply_fn = hk.without_apply_rng(hk.transform(f))

In [11]:
params = init_fn(jax.random.PRNGKey(42), data)

In [15]:
jax.jit(apply_fn)(params, data).shape

(10, 3, 10)

In [14]:
jax.jit(apply_fn)(params, data_sub).shape

(10, 2, 10)

In [11]:
from probjax.nn.tokenizer import StructuredTokenizer

In [ ]:
jax.jit(apply_fn)(params, node_id, values)

TracerIntegerConversionError: The __index__() method was called on traced array with shape int32[].
The error occurred while tracing the function apply_fn at /root/miniconda3/envs/probjax/lib/python3.10/site-packages/haiku/_src/multi_transform.py:312 for jit. This concrete value was not available in Python because it depends on the value of the argument ids.
See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.TracerIntegerConversionError

In [11]:
#with jax.disable_jit():
print(apply_fn(params, node_id, values).shape)

ValueError: 'structured_tokenizer/linear_1/w' with retrieved shape (10, 5) does not match shape=[1, 5] dtype=dtype('float32')

In [ ]:
def structured_transformer_model(
    num_nodes: int,
    token_dim: int =40,
    condition_token_dim: int =10,
    condition_token_init_scale: int =0.01,
    condition_token_init_mean: int=0.0,
    condition_mode: str="concat",
    time_embedding_dim: int=128,
    num_heads: int=4,
    num_layers: int=6,
    attn_size: int=5,
    widening_factor: int=4,
    num_hidden_layers: int=1,
    act=jax.nn.gelu,
    value_embeding_builder: Optional[Callable] = None,
    node_embeding_builder: Optional[Callable] = None,
    node_meta_data_embeding_builder: Optional[Callable] = None,
    skip_connection_attn: bool=True,
    skip_connection_mlp: bool=True,
    layer_norm: bool=True,
    output_scale_fn=None,
    base_mask=None,
    **kwargs,
):
    if output_scale_fn is None:
        output_scale_fn = lambda t, x: x

    if condition_mode == "concat":
        condition_token_dim = condition_token_dim
    elif condition_mode == "add":
        token_dim = token_dim + condition_token_dim
        condition_token_dim = token_dim
    elif condition_mode == "none":
        token_dim = token_dim + condition_token_dim
        condition_token_dim = 0

    def model(t, data, data_id, condition_mask, meta_data=None, edge_mask=base_mask):
        current_nodes = len(data)
        data_dim = jax.tree_map(lambda x: x.shape[-1], data)
        data_id = data_id.reshape(current_nodes)
        condition_mask = condition_mask.reshape(-1, current_nodes)

        tokenizer = StructuredTokenizer(token_dim, num_nodes, value_embeding_builder, node_embeding_builder, node_meta_data_embeding_builder)
        time_embeder = GaussianFourierEmbedding(time_embedding_dim)

        # Embedding
        tokens = tokenizer(data_id, data, meta_data)
        time = time_embeder(t[..., None])

        # Conditioning
        if condition_mode != "none":
            condition_token = hk.get_parameter(
                "condition_token",
                shape=[1, 1, condition_token_dim],
                init=hk.initializers.RandomNormal(
                    condition_token_init_scale, condition_token_init_mean
                ),
            )
            condition_mask = condition_mask.reshape(-1, current_nodes, 1)
            condition_token = condition_mask * condition_token
            if condition_mode == "add":
                tokens = tokens + condition_token
            elif condition_mode == "concat":
                condition_token = jnp.broadcast_to(
                    condition_token, tokens.shape[:-1] + (condition_token_dim,)
                )
                tokens = jnp.concatenate([tokens, condition_token], -1)

        # Forward pass

        model = Transformer(
            num_heads=num_heads,
            num_layers=num_layers,
            attn_size=attn_size,
            widening_factor=widening_factor,
            num_hidden_layers=num_hidden_layers,
            act=act,
            skip_connection_attn=skip_connection_attn,
            skip_connection_mlp=skip_connection_mlp,
 #           layer_norm=layer_norm,
        )

        h = model(tokens, context=time, mask=edge_mask)
        output_fn = [hk.Linear(d) for d in data_dim]
        out = jnp.concatenate([output_fn[i](h[..., i, :]) for i in range(len(data_dim))], axis=-1)
        out = output_scale_fn(t, out)
        return out

    init_fn, model_fn = hk.without_apply_rng(hk.transform(model))
    return init_fn, model_fn

In [ ]:
init_fn, model_fn = structured_transformer_model(10)

In [ ]:
params = init_fn(jax.random.PRNGKey(0), jnp.ones((10,1)),(jnp.ones((10,3)), jnp.ones((10,10))),  jnp.arange(2), jnp.array([True, False]))

In [ ]:
model_fn(params, jnp.ones((10,1)),(jnp.ones((10,3)), jnp.ones((10,10)))[:1],  jnp.arange(2)[:1], jnp.array([True, False])[:1]).shape

(10, 3)